# Stage 2 — Adaptive Synthetic Budget Allocation for Low-Data Image Classification

This notebook investigates whether class-aware synthetic budget allocation outperforms uniform augmentation under fixed compute. We measure class-level utility of diffusion-generated images, build a predictive model, and test multiple allocation policies.

| Pipeline | Data | Purpose |
|----------|------|---------|
| **Baseline** | 5% real (25 img/class) | Low-data floor |
| **Uniform DiffusionBoost** | 5% real + uniform synthetic | Standard augmentation |
| **Adaptive DiffusionBoost** | 5% real + class-aware allocation (same budget) | Targeted augmentation |
| **Ceiling** | 100% real (500 img/class) | Full-data upper bound |

All pipelines run on both **ResNet-18** and **MobileNetV3**. Synthetic budgets: 5x, 10x, 15x real data per class.

In [ ]:
import os, sys, random, csv, time, json
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.cuda.amp import GradScaler, autocast
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms, models
from PIL import Image
from scipy import linalg as scipy_linalg
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name()}  |  VRAM: {torch.cuda.get_device_properties(0).total_mem/1e9:.1f} GB")

PROJECT_ROOT = Path("..").resolve()
DATA_ROOT    = PROJECT_ROOT / "data"
TINY_ROOT    = DATA_ROOT / "raw" / "tiny-imagenet-200"
SUBSET_INDEX = DATA_ROOT / "tiny_imagenet_5pct" / "train_index.csv"
SYNTH_ROOT   = DATA_ROOT / "synthetic_sd"
FIG          = PROJECT_ROOT / "figures" / "stage2"
CKP          = PROJECT_ROOT / "checkpoints"
for d in [FIG, CKP]: d.mkdir(parents=True, exist_ok=True)

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
BS, NW, NC, IMG = 64, 0, 200, 224
EPOCHS, LR, PATIENCE = 30, 3e-4, 7
REAL_PER_CLASS = 25
SYNTH_RATIOS = [5, 10, 15]
MAX_SYNTH_PER_CLASS = REAL_PER_CLASS * max(SYNTH_RATIOS)

## 1. Datasets, Transforms, and Models

In [ ]:
class SubsetDS(Dataset):
    def __init__(self, root, csv_path, transform=None, c2i=None):
        self.root, self.transform = Path(root), transform
        with Path(csv_path).open() as f: self.entries = list(csv.DictReader(f))
        if c2i is None:
            cids = sorted({r["class_id"] for r in self.entries})
            self.c2i = {c: i for i, c in enumerate(cids)}
        else: self.c2i = c2i
    def __len__(self): return len(self.entries)
    def __getitem__(self, i):
        e = self.entries[i]
        img = Image.open(self.root / e["image_path"]).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.c2i[e["class_id"]], e["class_id"]

class FullTrainDS(Dataset):
    def __init__(self, root, transform=None, c2i=None):
        tr = Path(root) / "train"; self.transform = transform
        self.samples = []
        for d in sorted(tr.iterdir()):
            if not d.is_dir(): continue
            for p in sorted((d/"images").iterdir()):
                if p.suffix.lower() in {".jpeg",".jpg",".png"}: self.samples.append((p, d.name))
        if c2i is None:
            cids = sorted({c for _,c in self.samples})
            self.c2i = {c:i for i,c in enumerate(cids)}
        else: self.c2i = c2i
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, c = self.samples[i]
        img = Image.open(p).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.c2i[c], c

class ValDS(Dataset):
    def __init__(self, root, transform=None, c2i=None):
        self.root, self.transform = Path(root), transform
        ann = self.root / "val" / "val_annotations.txt"
        self.samples = []
        with ann.open() as f:
            for line in f:
                p = line.strip().split()
                if len(p)>=2: self.samples.append((self.root/"val"/"images"/p[0], p[1]))
        if c2i is None:
            cids = sorted({c for _,c in self.samples})
            self.c2i = {c:i for i,c in enumerate(cids)}
        else: self.c2i = c2i
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, c = self.samples[i]
        img = Image.open(p).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.c2i[c], c

class SynthDS(Dataset):
    """Loads synthetic images with per-class budget control."""
    def __init__(self, root, transform=None, c2i=None, alloc=None, max_per_class=None):
        self.root, self.transform = Path(root), transform
        self.samples = []
        for d in sorted(self.root.iterdir()):
            if not d.is_dir(): continue
            cid = d.name
            limit = max_per_class
            if alloc is not None and c2i is not None and cid in c2i:
                limit = int(alloc[c2i[cid]])
            imgs = sorted(p for p in d.iterdir() if p.suffix.lower() in {".jpeg",".jpg",".png"})
            if limit is not None: imgs = imgs[:limit]
            for p in imgs: self.samples.append((p, cid))
        if c2i is None:
            cids = sorted({c for _,c in self.samples})
            self.c2i = {c:i for i,c in enumerate(cids)}
        else: self.c2i = c2i
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        p, c = self.samples[i]
        img = Image.open(p).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, self.c2i[c], c

class CombinedDS(Dataset):
    """Interleaves real and synthetic samples."""
    def __init__(self, real, synth):
        self.real, self.synth = real, synth
    def __len__(self): return len(self.real) + len(self.synth)
    def __getitem__(self, i):
        if i < len(self.real): return self.real[i]
        return self.synth[i - len(self.real)]

In [ ]:
train_tf = transforms.Compose([transforms.Resize(256), transforms.RandomResizedCrop(IMG),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
val_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG),
    transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
raw_tf = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(IMG), transforms.ToTensor()])

base_ds = SubsetDS(TINY_ROOT/"train", SUBSET_INDEX, transform=train_tf)
C2I = base_ds.c2i
I2C = {v:k for k,v in C2I.items()}
val_ds = ValDS(TINY_ROOT, transform=val_tf, c2i=C2I)
val_loader = DataLoader(val_ds, batch_size=BS, shuffle=False, num_workers=NW)

W2L = {}
wp = TINY_ROOT / "words.txt"
if wp.exists():
    with wp.open() as f:
        for line in f:
            parts = line.strip().split("\t")
            if len(parts)==2: W2L[parts[0]] = parts[1].split(",")[0].strip()

print(f"Classes: {NC}  |  Real/class: {REAL_PER_CLASS}  |  Val: {len(val_ds):,}")

In [ ]:
class TinyModel(nn.Module):
    """ResNet-18 or MobileNetV3 adapted for 200-class classification."""
    def __init__(self, arch="resnet18", num_classes=200):
        super().__init__()
        self.arch = arch
        if arch == "resnet18":
            self.bb = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.bb.fc = nn.Linear(self.bb.fc.in_features, num_classes)
        elif arch == "mobilenet_v3":
            self.bb = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V1)
            self.bb.classifier[3] = nn.Linear(self.bb.classifier[3].in_features, num_classes)
        else: raise ValueError(f"Unknown: {arch}")

    def forward(self, x, return_features=False):
        if not return_features:
            return self.bb(x)
        if self.arch == "resnet18":
            x = self.bb.conv1(x); x = self.bb.bn1(x); x = self.bb.relu(x); x = self.bb.maxpool(x)
            x = self.bb.layer1(x); x = self.bb.layer2(x); x = self.bb.layer3(x); x = self.bb.layer4(x)
            x = self.bb.avgpool(x); feat = torch.flatten(x, 1)
            return self.bb.fc(feat), feat
        else:
            feat = self.bb.features(x)
            feat = nn.functional.adaptive_avg_pool2d(feat, 1).flatten(1)
            return self.bb.classifier(feat), feat

for a in ["resnet18", "mobilenet_v3"]:
    m = TinyModel(a); p = sum(p.numel() for p in m.parameters() if p.requires_grad)
    print(f"  {a}: {p:,} params")
del m

## 2. Synthetic Data Generation

In [ ]:
GENERATE = False

if GENERATE:
    from diffusers import StableDiffusionPipeline, DPMSolverMultistepScheduler
    TEMPLATES = [
        "a photo of a {l}, high quality, detailed, realistic",
        "a photograph of a {l}, natural lighting, sharp focus",
        "a clear image of a {l}, professional photography",
        "a {l}, realistic, well-lit, high resolution",
    ]
    NEG = "blurry, low quality, distorted, watermark, text, cartoon, drawing"
    cids = sorted([d.name for d in (TINY_ROOT/"train").iterdir() if d.is_dir()])
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16, safety_checker=None)
    pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to(DEVICE); pipe.set_progress_bar_config(disable=True)
    if hasattr(pipe, "enable_attention_slicing"): pipe.enable_attention_slicing()
    gen = torch.Generator(device=DEVICE).manual_seed(SEED)
    SYNTH_ROOT.mkdir(parents=True, exist_ok=True)
    for cid in tqdm(cids, desc="Generating"):
        lab = W2L.get(cid, cid); cd = SYNTH_ROOT/cid; cd.mkdir(exist_ok=True)
        ex = len(list(cd.glob("*.png")))
        if ex >= MAX_SYNTH_PER_CLASS: continue
        rem = MAX_SYNTH_PER_CLASS - ex
        for b in range(0, rem, 4):
            n = min(4, rem-b)
            prompts = [TEMPLATES[(ex+b+j)%len(TEMPLATES)].format(l=lab) for j in range(n)]
            with torch.no_grad():
                imgs = pipe(prompts, negative_prompt=[NEG]*n, num_inference_steps=25,
                            guidance_scale=7.5, height=512, width=512, generator=gen).images
            for j, im in enumerate(imgs): im.save(cd/f"img_{ex+b+j:04d}.png")
    del pipe; torch.cuda.empty_cache()
    print("Generation complete.")
else:
    sc = sum(1 for _ in SYNTH_ROOT.rglob("*.png")) if SYNTH_ROOT.exists() else 0
    print(f"Cached synthetic images: {sc:,}")
    if sc == 0: print("WARNING: No synthetic data found. Set GENERATE=True or use the CLI.")

## 3. Synthetic Image Quality (FID)

In [ ]:
def get_inception_features(loader, device=DEVICE, max_samples=5000):
    inception = models.inception_v3(weights=models.Inception_V3_Weights.IMAGENET1K_V1)
    inception.fc = nn.Identity()
    inception.eval().to(device)
    resize = transforms.Resize((299, 299), antialias=True)
    feats = []; n = 0
    with torch.no_grad():
        for imgs, _, _ in loader:
            f = inception(resize(imgs).to(device))
            if isinstance(f, tuple): f = f[0]
            feats.append(f.cpu().numpy()); n += imgs.size(0)
            if n >= max_samples: break
    del inception; torch.cuda.empty_cache()
    return np.concatenate(feats)[:max_samples]

def compute_fid(feats_a, feats_b):
    mu_a, sig_a = feats_a.mean(0), np.cov(feats_a, rowvar=False)
    mu_b, sig_b = feats_b.mean(0), np.cov(feats_b, rowvar=False)
    d = mu_a - mu_b
    cm = scipy_linalg.sqrtm(sig_a @ sig_b)
    if np.iscomplexobj(cm): cm = cm.real
    return float(d @ d + np.trace(sig_a + sig_b - 2*cm))

has_synth = SYNTH_ROOT.exists() and any(SYNTH_ROOT.iterdir()) if SYNTH_ROOT.exists() else False
if has_synth:
    print("Computing FID...")
    real_fid_ds = SubsetDS(TINY_ROOT/"train", SUBSET_INDEX, transform=val_tf, c2i=C2I)
    synth_fid_ds = SynthDS(SYNTH_ROOT, transform=val_tf, c2i=C2I)
    f_r = get_inception_features(DataLoader(real_fid_ds, BS, num_workers=NW))
    f_s = get_inception_features(DataLoader(synth_fid_ds, BS, num_workers=NW))
    fid_score = compute_fid(f_r, f_s)
    print(f"FID (real vs synthetic): {fid_score:.2f}")
else:
    fid_score = None
    print("Skipping FID (no synthetic data)")

## 4. Training Utilities

In [ ]:
def acc_fn(out, tgt):
    return out.argmax(1).eq(tgt).float().mean().item() * 100

class EarlyStopping:
    def __init__(self, patience=7):
        self.p, self.c, self.best, self.stop = patience, 0, None, False
    def step(self, s):
        if self.best is None or s > self.best: self.best, self.c = s, 0
        else:
            self.c += 1
            if self.c >= self.p: self.stop = True
        return self.stop

def train_pipeline(name, train_ds, val_loader, arch="resnet18",
                   epochs=EPOCHS, lr=LR, patience=PATIENCE,
                   ckpt=None, skip_if_exists=True):
    if ckpt and ckpt.exists() and skip_if_exists:
        print(f"\n[{name}] Loading checkpoint: {ckpt.name}")
        m = TinyModel(arch, NC).to(DEVICE)
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        hp = ckpt.with_suffix(".json")
        h = json.loads(hp.read_text()) if hp.exists() else None
        return m, h

    print(f"\n{'='*60}\n  [{name}] {arch} | {len(train_ds):,} samples | max {epochs} epochs\n{'='*60}")
    loader = DataLoader(train_ds, batch_size=BS, shuffle=True, num_workers=NW, pin_memory=True)
    m = TinyModel(arch, NC).to(DEVICE)
    crit = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=0.01)
    sched = CosineAnnealingLR(opt, T_max=epochs)
    scaler = GradScaler(enabled=torch.cuda.is_available())
    es = EarlyStopping(patience)
    h = {"train_loss":[], "train_acc":[], "val_loss":[], "val_acc":[], "lr":[]}
    best = 0.0
    for ep in range(1, epochs+1):
        h["lr"].append(opt.param_groups[0]["lr"])
        m.train(); rl,rc,rt = 0.,0.,0
        for imgs,tgts,_ in tqdm(loader, desc=f"[{name}] E{ep}", leave=False):
            imgs,tgts = imgs.to(DEVICE),tgts.to(DEVICE); opt.zero_grad()
            with autocast(enabled=torch.cuda.is_available()):
                out=m(imgs); loss=crit(out,tgts)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            b=tgts.size(0); rl+=loss.item()*b; rc+=acc_fn(out,tgts)/100*b; rt+=b
        m.eval(); vl,vc,vt = 0.,0.,0
        with torch.no_grad():
            for imgs,tgts,_ in tqdm(val_loader, desc="Val", leave=False):
                imgs,tgts = imgs.to(DEVICE),tgts.to(DEVICE)
                out=m(imgs); loss=crit(out,tgts)
                b=tgts.size(0); vl+=loss.item()*b; vc+=acc_fn(out,tgts)/100*b; vt+=b
        tl,ta,vloss,va = rl/rt,rc/rt,vl/vt,vc/vt
        h["train_loss"].append(tl); h["train_acc"].append(ta)
        h["val_loss"].append(vloss); h["val_acc"].append(va)
        sched.step()
        tag = ""
        if va > best:
            best = va
            if ckpt: ckpt.parent.mkdir(parents=True, exist_ok=True); torch.save(m.state_dict(), ckpt); tag=" *"
        print(f"  E{ep:02d} TL:{tl:.4f} TA:{ta*100:.1f}% VL:{vloss:.4f} VA:{va*100:.1f}%{tag}")
        if es.step(va): print(f"  Early stop at epoch {ep}"); break
    if ckpt and ckpt.exists():
        m.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        ckpt.with_suffix(".json").write_text(json.dumps(h, indent=2))
    return m, h

## 5. Baseline Training (Both Architectures)

In [ ]:
baseline_ds = SubsetDS(TINY_ROOT/"train", SUBSET_INDEX, transform=train_tf, c2i=C2I)
ceiling_ds  = FullTrainDS(TINY_ROOT, transform=train_tf, c2i=C2I)

m_r18_base, h_r18_base = train_pipeline("R18-Baseline", baseline_ds, val_loader, "resnet18", ckpt=CKP/"s2_r18_baseline.pth")
m_mv3_base, h_mv3_base = train_pipeline("MV3-Baseline", baseline_ds, val_loader, "mobilenet_v3", ckpt=CKP/"s2_mv3_baseline.pth")

## 6. Class-Level Diagnostics

Extract per-class accuracy, confidence, entropy, and feature geometry from the baseline models. These properties will inform the allocation policies.

In [ ]:
def per_class_diagnostics(model, loader, n_classes=NC):
    """Per-class accuracy, mean confidence, mean entropy, and per-sample logits."""
    model.eval()
    correct = torch.zeros(n_classes); total = torch.zeros(n_classes)
    conf_sum = torch.zeros(n_classes); ent_sum = torch.zeros(n_classes)
    all_logits, all_labels = [], []
    with torch.no_grad():
        for imgs, tgts, _ in loader:
            logits = model(imgs.to(DEVICE)).cpu()
            all_logits.append(logits); all_labels.append(tgts)
            probs = torch.softmax(logits, 1)
            preds = probs.argmax(1)
            max_conf, _ = probs.max(1)
            entropy = -(probs * probs.clamp(min=1e-8).log()).sum(1)
            for c in range(n_classes):
                mask = tgts == c
                total[c] += mask.sum()
                correct[c] += (preds[mask] == c).sum()
                conf_sum[c] += max_conf[mask].sum()
                ent_sum[c] += entropy[mask].sum()
    pca = (correct / total.clamp(min=1)).numpy()
    pcc = (conf_sum / total.clamp(min=1)).numpy()
    pce = (ent_sum / total.clamp(min=1)).numpy()
    return pca, pcc, pce, torch.cat(all_logits), torch.cat(all_labels)

def extract_features(model, loader):
    model.eval(); fs, ls = [], []
    with torch.no_grad():
        for imgs, tgts, _ in loader:
            _, f = model(imgs.to(DEVICE), return_features=True)
            fs.append(f.cpu().numpy()); ls.append(tgts.numpy())
    return np.concatenate(fs), np.concatenate(ls)

def feature_geometry(feats, labels, n_classes=NC):
    """Per-class compactness (mean dist to centroid) and separation (dist to nearest other centroid)."""
    centroids = np.zeros((n_classes, feats.shape[1]))
    compactness = np.zeros(n_classes)
    for c in range(n_classes):
        mask = labels == c
        if mask.sum() == 0: continue
        cf = feats[mask]
        centroids[c] = cf.mean(0)
        compactness[c] = np.linalg.norm(cf - centroids[c], axis=1).mean()
    separation = np.full(n_classes, np.inf)
    for c in range(n_classes):
        for j in range(n_classes):
            if c == j: continue
            d = np.linalg.norm(centroids[c] - centroids[j])
            if d < separation[c]: separation[c] = d
    margin = separation - compactness
    return compactness, separation, margin, centroids

print("Extracting R18 baseline diagnostics...")
r18_pca, r18_pcc, r18_pce, r18_logits, r18_labels = per_class_diagnostics(m_r18_base, val_loader)
r18_feats, r18_flabels = extract_features(m_r18_base, val_loader)
r18_compact, r18_sep, r18_margin, r18_centroids = feature_geometry(r18_feats, r18_flabels)
print(f"  Mean per-class acc: {r18_pca.mean()*100:.1f}%  |  Mean entropy: {r18_pce.mean():.2f}")

print("Extracting MV3 baseline diagnostics...")
mv3_pca, mv3_pcc, mv3_pce, mv3_logits, mv3_labels = per_class_diagnostics(m_mv3_base, val_loader)
print(f"  Mean per-class acc: {mv3_pca.mean()*100:.1f}%  |  Mean entropy: {mv3_pce.mean():.2f}")

## 7. Synthetic Quality per Class

In [ ]:
def synth_quality_per_class(model, real_centroids, synth_root, c2i, transform, n_classes=NC):
    """Per-class synthetic quality: mean feature distance to real centroid & intra-class diversity."""
    model.eval()
    fidelity = np.full(n_classes, np.nan)
    diversity = np.full(n_classes, np.nan)
    for d in sorted(synth_root.iterdir()):
        if not d.is_dir() or d.name not in c2i: continue
        ci = c2i[d.name]
        imgs_paths = sorted(p for p in d.iterdir() if p.suffix.lower() in {".jpeg",".jpg",".png"})[:MAX_SYNTH_PER_CLASS]
        if not imgs_paths: continue
        feats = []
        for batch_start in range(0, len(imgs_paths), BS):
            batch_paths = imgs_paths[batch_start:batch_start+BS]
            batch_imgs = torch.stack([transform(Image.open(p).convert("RGB")) for p in batch_paths])
            with torch.no_grad():
                _, f = model(batch_imgs.to(DEVICE), return_features=True)
            feats.append(f.cpu().numpy())
        feats = np.concatenate(feats)
        fidelity[ci] = np.linalg.norm(feats - real_centroids[ci], axis=1).mean()
        if len(feats) > 1:
            cent = feats.mean(0)
            diversity[ci] = np.linalg.norm(feats - cent, axis=1).mean()
    return fidelity, diversity

if has_synth:
    print("Computing per-class synthetic quality (R18 features)...")
    synth_fidelity, synth_diversity = synth_quality_per_class(
        m_r18_base, r18_centroids, SYNTH_ROOT, C2I, val_tf)
    valid = ~np.isnan(synth_fidelity)
    print(f"  Mean fidelity (lower=closer): {synth_fidelity[valid].mean():.2f}")
    print(f"  Mean diversity: {synth_diversity[valid].mean():.2f}")
else:
    synth_fidelity = synth_diversity = np.full(NC, np.nan)
    print("Skipping synthetic quality (no data)")

## 8. Allocation Policies & Dataset Construction

In [ ]:
def allocate_budget(strategy, total_budget, scores=None, cap=MAX_SYNTH_PER_CLASS, floor=5):
    """Distribute total_budget across NC classes. Returns integer allocation array."""
    n = NC
    if strategy == "uniform":
        raw = np.full(n, total_budget / n)
    elif strategy == "hard_class":
        difficulty = 1.0 - scores
        difficulty = np.maximum(difficulty, 0)
        weights = difficulty / difficulty.sum() if difficulty.sum() > 0 else np.ones(n)/n
        raw = weights * total_budget
    elif strategy == "uncertainty":
        weights = scores / scores.sum() if scores.sum() > 0 else np.ones(n)/n
        raw = weights * total_budget
    elif strategy == "fidelity":
        inv_dist = 1.0 / np.maximum(scores, 1e-6)
        inv_dist[np.isnan(scores)] = 0
        weights = inv_dist / inv_dist.sum() if inv_dist.sum() > 0 else np.ones(n)/n
        raw = weights * total_budget
    elif strategy == "predicted_utility":
        pos = np.maximum(scores, 0)
        weights = pos / pos.sum() if pos.sum() > 0 else np.ones(n)/n
        raw = weights * total_budget
    else:
        raise ValueError(f"Unknown: {strategy}")
    alloc = np.clip(np.round(raw).astype(int), floor, cap)
    # Adjust to match total budget
    diff = int(total_budget) - alloc.sum()
    idxs = np.argsort(raw)[::-1] if diff > 0 else np.argsort(raw)
    for i in idxs:
        if diff == 0: break
        adj = min(diff, cap - alloc[i]) if diff > 0 else max(diff, floor - alloc[i])
        alloc[i] += adj; diff -= adj
    return alloc

def build_diffboost_ds(alloc):
    """Create CombinedDS with per-class synthetic allocation."""
    sds = SynthDS(SYNTH_ROOT, transform=train_tf, c2i=C2I, alloc=alloc)
    return CombinedDS(baseline_ds, sds)

# Uniform datasets at each scaling ratio
uniform_datasets = {}
if has_synth:
    for ratio in SYNTH_RATIOS:
        budget_per_class = REAL_PER_CLASS * ratio
        total = budget_per_class * NC
        alloc = allocate_budget("uniform", total, cap=budget_per_class)
        uniform_datasets[ratio] = build_diffboost_ds(alloc)
        print(f"  Uniform {ratio}x: {len(uniform_datasets[ratio]):,} total samples")

print(f"Baseline: {len(baseline_ds):,}  |  Ceiling: {len(ceiling_ds):,}")

## 9. Uniform Scaling Study (ResNet-18)

In [ ]:
uniform_models_r18, uniform_hists_r18 = {}, {}
for ratio, ds in uniform_datasets.items():
    m, h = train_pipeline(f"R18-Uniform-{ratio}x", ds, val_loader, "resnet18",
                          ckpt=CKP/f"s2_r18_uniform_{ratio}x.pth")
    uniform_models_r18[ratio] = m
    uniform_hists_r18[ratio] = h

## 10. Measure Per-Class Utility & Build Utility Prediction Model

Using the uniform 15x model, compute per-class utility = accuracy_c(DiffBoost) - accuracy_c(Baseline). Then predict utility from class-level features.

In [ ]:
if has_synth and max(SYNTH_RATIOS) in uniform_models_r18:
    max_ratio = max(SYNTH_RATIOS)
    db_pca, _, _, _, _ = per_class_diagnostics(uniform_models_r18[max_ratio], val_loader)
    utility = db_pca - r18_pca

    n_helped = (utility > 0.001).sum()
    n_harmed = (utility < -0.001).sum()
    n_neutral = NC - n_helped - n_harmed
    print(f"Per-class utility (uniform {max_ratio}x vs baseline):")
    print(f"  Helped: {n_helped}  |  Harmed: {n_harmed}  |  Neutral: {n_neutral}")
    print(f"  Mean utility: {utility.mean()*100:.2f}pp  |  Std: {utility.std()*100:.2f}pp")

    # Build feature matrix for utility prediction
    valid = ~np.isnan(synth_fidelity)
    X_util = np.column_stack([
        r18_pca,
        r18_pcc,
        r18_pce,
        r18_compact,
        r18_sep,
        r18_margin,
        np.nan_to_num(synth_fidelity, nan=synth_fidelity[valid].mean() if valid.any() else 0),
        np.nan_to_num(synth_diversity, nan=synth_diversity[valid].mean() if valid.any() else 0),
    ])
    y_util = utility
    feature_names = ["baseline_acc", "confidence", "entropy", "compactness",
                     "separation", "margin", "synth_fidelity", "synth_diversity"]

    scaler_util = StandardScaler()
    X_scaled = scaler_util.fit_transform(X_util)
    ridge = Ridge(alpha=1.0)
    cv_scores = cross_val_score(ridge, X_scaled, y_util, cv=10, scoring="r2")
    print(f"\nUtility prediction (Ridge, 10-fold CV): R² = {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

    ridge.fit(X_scaled, y_util)
    print("\nFeature importances (Ridge coefficients):")
    for fn, coef in sorted(zip(feature_names, ridge.coef_), key=lambda x: abs(x[1]), reverse=True):
        print(f"  {fn:20s} {coef:+.4f}")

    predicted_utility = ridge.predict(X_scaled)
else:
    utility = np.zeros(NC)
    predicted_utility = np.zeros(NC)
    print("Utility analysis skipped (no synthetic data)")

## 11. Adaptive Allocation & Policy Training

In [ ]:
max_ratio = max(SYNTH_RATIOS)
total_budget_15x = REAL_PER_CLASS * max_ratio * NC

policies = {}
policy_models_r18 = {}
policy_hists_r18 = {}

if has_synth:
    # Uniform at 15x is already trained
    policies["uniform"] = allocate_budget("uniform", total_budget_15x)
    policy_models_r18["uniform"] = uniform_models_r18[max_ratio]
    policy_hists_r18["uniform"] = uniform_hists_r18[max_ratio]

    # Hard-class
    policies["hard_class"] = allocate_budget("hard_class", total_budget_15x, scores=r18_pca)
    ds_hc = build_diffboost_ds(policies["hard_class"])
    m_hc, h_hc = train_pipeline("R18-HardClass-15x", ds_hc, val_loader, "resnet18",
                                 ckpt=CKP/"s2_r18_hardclass_15x.pth")
    policy_models_r18["hard_class"] = m_hc
    policy_hists_r18["hard_class"] = h_hc

    # Uncertainty-based
    policies["uncertainty"] = allocate_budget("uncertainty", total_budget_15x, scores=r18_pce)
    ds_unc = build_diffboost_ds(policies["uncertainty"])
    m_unc, h_unc = train_pipeline("R18-Uncertainty-15x", ds_unc, val_loader, "resnet18",
                                   ckpt=CKP/"s2_r18_uncertainty_15x.pth")
    policy_models_r18["uncertainty"] = m_unc
    policy_hists_r18["uncertainty"] = h_unc

    # Predicted-utility
    policies["predicted_utility"] = allocate_budget("predicted_utility", total_budget_15x, scores=predicted_utility)
    ds_pu = build_diffboost_ds(policies["predicted_utility"])
    m_pu, h_pu = train_pipeline("R18-PredUtil-15x", ds_pu, val_loader, "resnet18",
                                 ckpt=CKP/"s2_r18_predutil_15x.pth")
    policy_models_r18["predicted_utility"] = m_pu
    policy_hists_r18["predicted_utility"] = h_pu

    print("\nAllocation stats (images/class):")
    for pn, alloc in policies.items():
        print(f"  {pn:20s}  mean={alloc.mean():.0f}  min={alloc.min()}  max={alloc.max()}  total={alloc.sum()}")
else:
    print("Policy training skipped")

## 12. Ceiling Training

In [ ]:
m_r18_ceil, h_r18_ceil = train_pipeline("R18-Ceiling", ceiling_ds, val_loader, "resnet18", ckpt=CKP/"s2_r18_ceiling.pth")
m_mv3_ceil, h_mv3_ceil = train_pipeline("MV3-Ceiling", ceiling_ds, val_loader, "mobilenet_v3", ckpt=CKP/"s2_mv3_ceiling.pth")

## 13. Cross-Backbone Policy Transfer (MobileNetV3)

The allocation policy was designed using ResNet-18 class properties. Does it transfer to MobileNetV3?

In [ ]:
if has_synth:
    # Uniform 15x on MV3
    ds_mv3_unif = build_diffboost_ds(policies["uniform"])
    m_mv3_unif, h_mv3_unif = train_pipeline("MV3-Uniform-15x", ds_mv3_unif, val_loader, "mobilenet_v3",
                                             ckpt=CKP/"s2_mv3_uniform_15x.pth")

    # Best adaptive policy on MV3 (predicted-utility designed on R18)
    ds_mv3_pu = build_diffboost_ds(policies["predicted_utility"])
    m_mv3_pu, h_mv3_pu = train_pipeline("MV3-PredUtil-15x", ds_mv3_pu, val_loader, "mobilenet_v3",
                                         ckpt=CKP/"s2_mv3_predutil_15x.pth")
else:
    m_mv3_unif = m_mv3_pu = None
    h_mv3_unif = h_mv3_pu = None

---
## 14. Results

### 14a. Accuracy Metrics

In [ ]:
def full_metrics(model, loader):
    """Top-1, macro, and worst-20 accuracy."""
    model.eval()
    correct = torch.zeros(NC); total = torch.zeros(NC)
    with torch.no_grad():
        for imgs, tgts, _ in loader:
            preds = model(imgs.to(DEVICE)).argmax(1).cpu()
            for c in range(NC):
                mask = tgts == c
                total[c] += mask.sum()
                correct[c] += (preds[mask] == c).sum()
    pca = (correct / total.clamp(min=1)).numpy()
    top1 = correct.sum().item() / total.sum().item() * 100
    macro = pca.mean() * 100
    worst20 = np.sort(pca)[:20].mean() * 100
    return top1, macro, worst20, pca

print(f"{'Pipeline':30s} {'Arch':10s} {'Top-1':>7s} {'Macro':>7s} {'W-20':>7s}")
print("-"*65)

all_results = {}

for name, model, arch in [
    ("Baseline", m_r18_base, "R18"), ("Baseline", m_mv3_base, "MV3"),
    ("Ceiling", m_r18_ceil, "R18"), ("Ceiling", m_mv3_ceil, "MV3"),
]:
    t1, mc, w20, pca = full_metrics(model, val_loader)
    all_results[(name, arch)] = {"top1": t1, "macro": mc, "worst20": w20, "pca": pca}
    print(f"{name:30s} {arch:10s} {t1:6.2f}% {mc:6.2f}% {w20:6.2f}%")

# Uniform scaling
for ratio, model in uniform_models_r18.items():
    t1, mc, w20, pca = full_metrics(model, val_loader)
    all_results[(f"Uniform {ratio}x", "R18")] = {"top1": t1, "macro": mc, "worst20": w20, "pca": pca}
    print(f"{f'Uniform {ratio}x':30s} {'R18':10s} {t1:6.2f}% {mc:6.2f}% {w20:6.2f}%")

# Adaptive policies
for pname, model in policy_models_r18.items():
    if pname == "uniform": continue
    t1, mc, w20, pca = full_metrics(model, val_loader)
    all_results[(pname, "R18")] = {"top1": t1, "macro": mc, "worst20": w20, "pca": pca}
    print(f"{pname:30s} {'R18':10s} {t1:6.2f}% {mc:6.2f}% {w20:6.2f}%")

# MV3 transfer
for name, model in [("Uniform 15x", m_mv3_unif), ("PredUtil 15x", m_mv3_pu)]:
    if model is None: continue
    t1, mc, w20, pca = full_metrics(model, val_loader)
    all_results[(name, "MV3")] = {"top1": t1, "macro": mc, "worst20": w20, "pca": pca}
    print(f"{name:30s} {'MV3':10s} {t1:6.2f}% {mc:6.2f}% {w20:6.2f}%")

### 14b. Synthetic Data Scaling Curves

In [ ]:
if uniform_hists_r18:
    base_r = all_results[("Baseline", "R18")]
    ceil_r = all_results[("Ceiling", "R18")]
    ratios = sorted(uniform_models_r18.keys())
    accs = [all_results[(f"Uniform {r}x", "R18")]["top1"] for r in ratios]
    macros = [all_results[(f"Uniform {r}x", "R18")]["macro"] for r in ratios]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
    for ax, vals, ylabel, title in [
        (ax1, accs, "Top-1 Accuracy (%)", "Top-1 vs Synthetic Budget"),
        (ax2, macros, "Macro Accuracy (%)", "Macro vs Synthetic Budget"),
    ]:
        base_v = base_r["top1"] if "Top-1" in title else base_r["macro"]
        ceil_v = ceil_r["top1"] if "Top-1" in title else ceil_r["macro"]
        ax.axhline(base_v, color="#2196F3", ls="--", label=f"Baseline ({base_v:.1f}%)")
        ax.axhline(ceil_v, color="#4CAF50", ls="--", label=f"Ceiling ({ceil_v:.1f}%)")
        ax.plot(ratios, vals, "o-", color="#FF9800", lw=2, ms=8, label="Uniform DiffBoost")
        for r, v in zip(ratios, vals): ax.annotate(f"{v:.1f}%", (r, v), textcoords="offset points", xytext=(0,10), ha="center", fontsize=9)
        ax.set_xlabel("Synthetic ratio (Nx real)"); ax.set_ylabel(ylabel); ax.set_title(title)
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(FIG/"scaling_curves.png", dpi=150, bbox_inches="tight"); plt.show()

### 14c. Policy Comparison

In [ ]:
if policy_models_r18:
    pnames = list(policy_models_r18.keys())
    metrics = ["top1", "macro", "worst20"]
    data = {pn: [all_results.get((pn, "R18"), all_results.get((f"Uniform {max(SYNTH_RATIOS)}x", "R18"), {})).get(m, 0) for m in metrics] for pn in pnames}

    x = np.arange(len(metrics))
    w = 0.8 / len(pnames)
    fig, ax = plt.subplots(figsize=(10, 5))
    colors_p = ["#2196F3", "#FF9800", "#9C27B0", "#4CAF50"]
    for i, pn in enumerate(pnames):
        ax.bar(x + i*w, data[pn], w, label=pn, color=colors_p[i % len(colors_p)])
    ax.set_xticks(x + w*(len(pnames)-1)/2); ax.set_xticklabels(["Top-1", "Macro", "Worst-20"])
    ax.set_ylabel("Accuracy (%)"); ax.set_title(f"Allocation Policy Comparison (ResNet-18, {max(SYNTH_RATIOS)}x budget)")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout(); plt.savefig(FIG/"policy_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

### 14d. Per-Class Utility & Harm Detection

In [ ]:
if has_synth and "predicted_utility" in policy_models_r18:
    unif_pca = all_results.get((f"Uniform {max(SYNTH_RATIOS)}x", "R18"), {}).get("pca", r18_pca)
    adap_pca_res = all_results.get(("predicted_utility", "R18"), {}).get("pca", r18_pca)
    base_pca = all_results[("Baseline", "R18")]["pca"]

    delta_unif = (unif_pca - base_pca) * 100
    delta_adap = (adap_pca_res - base_pca) * 100
    order = np.argsort(delta_unif)

    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(range(NC), delta_unif[order], width=1.0, alpha=0.5, color="#2196F3", label="Uniform")
    ax.bar(range(NC), delta_adap[order], width=0.5, alpha=0.8, color="#FF9800", label="Adaptive")
    ax.axhline(0, color="black", lw=0.5)
    ax.set_xlabel("Class (sorted by uniform delta)"); ax.set_ylabel("Accuracy delta (pp)")
    n_helped_u = (delta_unif > 0.1).sum(); n_harmed_u = (delta_unif < -0.1).sum()
    n_helped_a = (delta_adap > 0.1).sum(); n_harmed_a = (delta_adap < -0.1).sum()
    ax.set_title(f"Per-Class Delta: Uniform ({n_helped_u} helped, {n_harmed_u} harmed) vs "
                 f"Adaptive ({n_helped_a} helped, {n_harmed_a} harmed)")
    ax.legend(); ax.grid(True, alpha=0.3, axis="y")
    plt.tight_layout(); plt.savefig(FIG/"perclass_delta.png", dpi=150, bbox_inches="tight"); plt.show()

    # Harm analysis
    harmed_unif = delta_unif < -0.1
    harmed_adap = delta_adap < -0.1
    print(f"\nHarm detection:")
    print(f"  Uniform: {harmed_unif.sum()} harmed classes  |  Adaptive: {harmed_adap.sum()} harmed classes")
    if harmed_unif.any():
        print(f"  Mean baseline acc of harmed classes (uniform): {r18_pca[harmed_unif].mean()*100:.1f}%")
        print(f"  Mean synth fidelity of harmed classes: {synth_fidelity[harmed_unif & ~np.isnan(synth_fidelity)].mean():.2f}")
    print(f"\nTop 5 harmed classes (uniform):")
    for i in np.argsort(delta_unif)[:5]:
        print(f"  {I2C[i]:12s} {W2L.get(I2C[i], '?'):25s} delta={delta_unif[i]:+.1f}pp  base_acc={r18_pca[i]*100:.0f}%")

### 14e. Calibration (ECE + Temperature Scaling)

In [ ]:
def collect_logits(model, loader):
    model.eval(); ls, ts = [], []
    with torch.no_grad():
        for imgs, tgts, _ in loader:
            ls.append(model(imgs.to(DEVICE)).cpu()); ts.append(tgts)
    return torch.cat(ls), torch.cat(ts)

def ece(logits, labels, n_bins=15):
    probs = torch.softmax(logits, 1).numpy()
    labs = labels.numpy(); confs, preds = probs.max(1), probs.argmax(1); accs = (preds==labs)
    bounds = np.linspace(0, 1, n_bins+1); e = 0.0; ba = []
    for i in range(n_bins):
        m = (confs > bounds[i]) & (confs <= bounds[i+1])
        if not m.any(): ba.append(0); continue
        a = accs[m].mean(); c = confs[m].mean()
        e += np.abs(a-c)*m.sum()/len(labs); ba.append(a)
    return e, bounds, ba

def find_temperature(logits, labels):
    def nll(T): return -torch.log_softmax(logits/T, 1)[range(len(labels)), labels].mean().item()
    return minimize_scalar(nll, bounds=(0.1, 10.0), method="bounded").x

calib_models = {"Baseline": m_r18_base, "Ceiling": m_r18_ceil}
if has_synth and max(SYNTH_RATIOS) in uniform_models_r18:
    calib_models[f"Uniform {max(SYNTH_RATIOS)}x"] = uniform_models_r18[max(SYNTH_RATIOS)]
if has_synth and "predicted_utility" in policy_models_r18:
    calib_models["Adaptive"] = policy_models_r18["predicted_utility"]

calib_results = {}
for nm, mdl in calib_models.items():
    logits, labels_c = collect_logits(mdl, val_loader)
    raw_e, bounds, ba = ece(logits, labels_c)
    T = find_temperature(logits, labels_c)
    cal_e, _, ba_cal = ece(logits / T, labels_c)
    calib_results[nm] = {"raw_ece": raw_e, "T": T, "cal_ece": cal_e, "bounds": bounds, "ba_raw": ba, "ba_cal": ba_cal}

n = len(calib_results)
fig, axes = plt.subplots(2, n, figsize=(5*n, 9))
if n == 1: axes = axes.reshape(2, 1)
for j, (nm, cr) in enumerate(calib_results.items()):
    centers = (cr["bounds"][:-1]+cr["bounds"][1:])/2
    for row, (ba, title) in enumerate([
        (cr["ba_raw"], f"{nm} Raw\nECE={cr['raw_ece']:.4f}"),
        (cr["ba_cal"], f"{nm} T={cr['T']:.2f}\nECE={cr['cal_ece']:.4f}"),
    ]):
        ax = axes[row, j]
        ax.plot([0,1],[0,1],"--",color="gray")
        ax.bar(centers, ba, width=1/15, alpha=0.7, edgecolor="black")
        ax.set_title(title, fontsize=10); ax.set_xlabel("Confidence"); ax.set_ylabel("Accuracy")
        ax.grid(True, alpha=0.3)
plt.suptitle("Reliability Diagrams: Raw vs Temperature-Scaled", fontsize=13)
plt.tight_layout(); plt.savefig(FIG/"reliability_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

print(f"{'Pipeline':25s} {'Raw ECE':>10s} {'Temp T':>8s} {'Cal ECE':>10s}")
print("-"*55)
for nm, cr in calib_results.items():
    print(f"{nm:25s} {cr['raw_ece']:10.4f} {cr['T']:8.2f} {cr['cal_ece']:10.4f}")

### 14f. Corruption Robustness

In [ ]:
val_raw_ds = ValDS(TINY_ROOT, transform=raw_tf, c2i=C2I)
val_raw_ld = DataLoader(val_raw_ds, batch_size=BS, shuffle=False, num_workers=NW)
mean_t = torch.tensor(IMAGENET_MEAN).view(1,3,1,1).to(DEVICE)
std_t  = torch.tensor(IMAGENET_STD).view(1,3,1,1).to(DEVICE)

corrs = {
    "Clean": None,
    "Noise": lambda x: (x + 0.1*torch.randn_like(x)).clamp(0,1),
    "Blur":  lambda x: transforms.GaussianBlur(5)(x),
    "Bright": lambda x: (x + 0.2).clamp(0,1),
}

def eval_rob(model, loader, cfn=None):
    model.eval(); c, t = 0, 0
    with torch.no_grad():
        for imgs, tgts, _ in loader:
            imgs = imgs.to(DEVICE)
            if cfn: imgs = cfn(imgs)
            imgs = (imgs - mean_t) / std_t
            c += (model(imgs).argmax(1) == tgts.to(DEVICE)).sum().item(); t += tgts.size(0)
    return c/t

rob_models = {"Baseline": m_r18_base, "Ceiling": m_r18_ceil}
if has_synth and max(SYNTH_RATIOS) in uniform_models_r18:
    rob_models[f"Uniform {max(SYNTH_RATIOS)}x"] = uniform_models_r18[max(SYNTH_RATIOS)]
for pn, mdl in policy_models_r18.items():
    if pn != "uniform": rob_models[pn] = mdl

print("Evaluating corruption robustness...")
rob = {}
for mn, mdl in rob_models.items():
    rob[mn] = {cn: eval_rob(mdl, val_raw_ld, cf) for cn, cf in corrs.items()}
    print(f"  {mn} done")

cnames = list(corrs.keys())
nm_list = list(rob.keys())
x = np.arange(len(cnames)); w = 0.8 / len(nm_list)
fig, ax = plt.subplots(figsize=(12, 5))
for i, mn in enumerate(nm_list):
    vals = [rob[mn][c]*100 for c in cnames]
    ax.bar(x + i*w, vals, w, label=mn)
ax.set_xticks(x + w*(len(nm_list)-1)/2); ax.set_xticklabels(cnames)
ax.set_ylabel("Accuracy (%)"); ax.set_title("Corruption Robustness by Policy")
ax.legend(fontsize=7); ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout(); plt.savefig(FIG/"robustness_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

### 14g. Feature Coverage Analysis

In [ ]:
coverage_models = {"Baseline": m_r18_base, "Ceiling": m_r18_ceil}
if has_synth and max(SYNTH_RATIOS) in uniform_models_r18:
    coverage_models[f"Uniform {max(SYNTH_RATIOS)}x"] = uniform_models_r18[max(SYNTH_RATIOS)]
if has_synth and "predicted_utility" in policy_models_r18:
    coverage_models["Adaptive"] = policy_models_r18["predicted_utility"]

print("Feature coverage analysis...")
coverage_res = {}
for nm, mdl in coverage_models.items():
    feats, labs = extract_features(mdl, val_loader)
    comp, sep, marg, _ = feature_geometry(feats, labs)
    coverage_res[nm] = {"compactness": comp.mean(), "separation": sep.mean(), "margin": marg.mean()}
    print(f"  {nm:20s}  compact={comp.mean():.2f}  sep={sep.mean():.2f}  margin={marg.mean():.2f}")

### 14h. Compute-Normalised Benefit

In [ ]:
if policy_models_r18:
    base_top1 = all_results[("Baseline", "R18")]["top1"]
    print(f"{'Policy':25s} {'Top1':>7s} {'Gain':>7s} {'Gain/1k imgs':>12s}")
    print("-"*55)
    for pn in policy_models_r18:
        key = (pn, "R18") if (pn, "R18") in all_results else (f"Uniform {max(SYNTH_RATIOS)}x", "R18")
        t1 = all_results[key]["top1"]
        gain = t1 - base_top1
        total_synth = policies[pn].sum()
        gain_per_1k = gain / (total_synth / 1000) if total_synth > 0 else 0
        print(f"{pn:25s} {t1:6.2f}% {gain:+6.2f}pp {gain_per_1k:+11.4f}pp")

### 14i. Representation Geometry (Eigenvalues, Linear Probe, CKA)

In [ ]:
repr_models = {"Baseline": m_r18_base, "Ceiling": m_r18_ceil}
if has_synth and max(SYNTH_RATIOS) in uniform_models_r18:
    repr_models[f"Uniform {max(SYNTH_RATIOS)}x"] = uniform_models_r18[max(SYNTH_RATIOS)]
if has_synth and "predicted_utility" in policy_models_r18:
    repr_models["Adaptive"] = policy_models_r18["predicted_utility"]

feat_dict = {}
for nm, mdl in repr_models.items():
    f, l = extract_features(mdl, val_loader)
    feat_dict[nm] = f
    print(f"  {nm}: {f.shape}")
labels_ref = l

# Eigenvalue spectrum
fig, ax = plt.subplots(figsize=(7, 4))
for nm, f in feat_dict.items():
    fc = f - f.mean(0, keepdims=True)
    ev = np.sort(np.linalg.eigvalsh(np.cov(fc, rowvar=False)))[::-1]
    ax.semilogy(ev, label=f"{nm} (eff rank={(ev>0.01*ev[0]).sum()})", lw=1.5)
ax.set_xlabel("Index"); ax.set_ylabel("Eigenvalue (log)")
ax.set_title("Covariance Eigenvalue Spectrum"); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig(FIG/"eigenvalue_comparison.png", dpi=150, bbox_inches="tight"); plt.show()

# Linear probe
n = len(labels_ref)
perm = np.random.RandomState(42).permutation(n)
sp = int(0.8*n); tri, tei = perm[:sp], perm[sp:]
for nm, f in feat_dict.items():
    clf = LogisticRegression(max_iter=500, solver="lbfgs", multi_class="multinomial", n_jobs=-1)
    clf.fit(f[tri], labels_ref[tri])
    print(f"  {nm} probe: {clf.score(f[tei], labels_ref[tei])*100:.2f}%")

# CKA
def lin_cka(X, Y):
    def hsic(K, L):
        n = K.shape[0]; H = np.eye(n) - np.ones((n,n))/n
        return float(np.trace(K@H@L@H) / ((n-1)**2))
    Kx, Ky = X@X.T, Y@Y.T
    return hsic(Kx, Ky) / np.sqrt(hsic(Kx, Kx)*hsic(Ky, Ky))

names_cka = list(feat_dict.keys()); nc = len(names_cka)
cka_idx = np.random.RandomState(42).choice(n, min(2000, n), replace=False)
cka_mat = np.ones((nc, nc))
for i in range(nc):
    for j in range(i+1, nc):
        v = lin_cka(feat_dict[names_cka[i]][cka_idx], feat_dict[names_cka[j]][cka_idx])
        cka_mat[i,j] = v; cka_mat[j,i] = v

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cka_mat, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(nc)); ax.set_xticklabels(names_cka, rotation=45, ha="right")
ax.set_yticks(range(nc)); ax.set_yticklabels(names_cka)
for i in range(nc):
    for j in range(nc): ax.text(j, i, f"{cka_mat[i,j]:.3f}", ha="center", va="center", fontsize=10)
plt.colorbar(im, ax=ax, label="CKA")
ax.set_title("Linear CKA Similarity")
plt.tight_layout(); plt.savefig(FIG/"cka_matrix.png", dpi=150, bbox_inches="tight"); plt.show()

### 14j. Cross-Backbone Transfer Validation

In [ ]:
print("\nCross-backbone transfer (policy designed on R18, applied to MV3):")
print(f"{'Pipeline':30s} {'Arch':6s} {'Top-1':>7s} {'Macro':>7s} {'Worst-20':>9s}")
print("-"*60)
for key in [("Baseline", "MV3"), ("Uniform 15x", "MV3"), ("PredUtil 15x", "MV3"), ("Ceiling", "MV3")]:
    if key in all_results:
        r = all_results[key]
        print(f"{key[0]:30s} {key[1]:6s} {r['top1']:6.2f}% {r['macro']:6.2f}% {r['worst20']:8.2f}%")

if ("Uniform 15x", "MV3") in all_results and ("PredUtil 15x", "MV3") in all_results:
    u = all_results[("Uniform 15x", "MV3")]
    a = all_results[("PredUtil 15x", "MV3")]
    print(f"\nTransfer gain (Adaptive - Uniform): "
          f"Top-1 {a['top1']-u['top1']:+.2f}pp  Macro {a['macro']-u['macro']:+.2f}pp  Worst-20 {a['worst20']-u['worst20']:+.2f}pp")

## 15. Summary

In [ ]:
print("\n" + "="*80)
print("  STAGE 2 — ADAPTIVE SYNTHETIC BUDGET ALLOCATION: FULL RESULTS")
print("="*80)

# Main comparison
print(f"\n{'Pipeline':30s} {'Arch':6s} {'Top-1':>7s} {'Macro':>7s} {'W-20':>7s}")
print("-"*65)
for key in sorted(all_results.keys(), key=lambda k: (k[1], k[0])):
    r = all_results[key]
    print(f"{key[0]:30s} {key[1]:6s} {r['top1']:6.2f}% {r['macro']:6.2f}% {r['worst20']:6.2f}%")

# Calibration
print(f"\n{'Pipeline':25s} {'Raw ECE':>10s} {'T':>6s} {'Cal ECE':>10s}")
print("-"*55)
for nm, cr in calib_results.items():
    print(f"{nm:25s} {cr['raw_ece']:10.4f} {cr['T']:6.2f} {cr['cal_ece']:10.4f}")

# FID
if fid_score is not None:
    print(f"\nFID (real vs synthetic): {fid_score:.2f}")

# Performance recovery
if ("Baseline", "R18") in all_results and ("Ceiling", "R18") in all_results:
    ba = all_results[("Baseline", "R18")]["top1"]
    ca = all_results[("Ceiling", "R18")]["top1"]
    gap = ca - ba
    print(f"\nR18 Ceiling-Baseline gap: {gap:.2f}pp")
    for key_prefix in [f"Uniform {max(SYNTH_RATIOS)}x", "predicted_utility"]:
        for k in all_results:
            if k[0] == key_prefix and k[1] == "R18":
                da = all_results[k]["top1"]
                rec = (da - ba) / gap * 100 if gap > 0 else 0
                print(f"  {k[0]:25s} recovery: {rec:.1f}% ({da:.2f}% vs {ba:.2f}%)")

print(f"\nFigures saved to: {FIG}")
print("="*80)